In [1]:

import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
%pip install openpyxl

# Add src directory to path
sys.path.append('../src')

from utils import load_tennis_data  # noqa: E402

# Try relative path inside repo
data_dir = Path('../data/processed')
rel_path = data_dir / 'match_core.xlsx'
print("Working dir:", Path.cwd())

if rel_path.exists():
    df = pd.read_excel(rel_path)
    print(f"Loaded {len(df)} rows from {rel_path}")
else:
    # Search for matching files in data/processed
    candidates = list(Path("data/processed").glob("match_core.*")) if Path("data/processed").exists() else []
    if candidates:
        chosen = candidates[0]
        print(f"Found file {chosen}, loading...")
        if chosen.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(chosen)
        else:
            df = pd.read_csv(chosen)
        print(f"Loaded {len(df)} rows from {chosen}")
    else:
        # Optional: use helper from src/utils.py which handles CSV loading
        try:
            csv_fallback = Path("data/processed/match_core.csv")
            if csv_fallback.exists():
                df = load_tennis_data(str(csv_fallback))
            else:
                raise FileNotFoundError(f"No match_core file found under data/processed. Checked: {rel_path}")
        except Exception as e:
            raise FileNotFoundError(f"Could not locate match_core file: {e}")


# Print the first few rows of the dataframe
print(df.head())

df = df.sort_values(by='Date', ascending=True)

print(df.head())

print(df.columns)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Working dir: /workspaces/tennis_betting_efficiency/notebooks
Loaded 24864 rows from ../data/processed/match_core.xlsx
   ATP Location   Tournament       Date       Series   Court Surface  \
0   65    Turin  Masters Cup 2024-11-17  Masters Cup  Indoor    Hard   
1   65    Turin  Masters Cup 2024-11-16  Masters Cup  Indoor    Hard   
2   65    Turin  Masters Cup 2024-11-16  Masters Cup  Indoor    Hard   
3   65    Turin  Masters Cup 2024-11-15  Masters Cup  Indoor    Hard   
4   65    Turin  Masters Cup 2024-11-15  Masters Cup  Indoor    Hard   

         Round  Best of     Winner  ...   PS_L_ip  PS_overround  PS_W_fair  \
0    The Final      3.0  Sinner J.  ...  0.190840      1.045541   0.817473   
1   Semifinals      3.0   Fritz T.  ...  0.675676      1.026553   0.341801   
2   Semifinals      3.0  Sinner J.  ..

In [2]:
import sys
import pandas as pd

sys.path.append('../src')
from ytd_stats import process_year_data, format_player_name

# -------------------------------
# 1. Build a single combined YTD dataset
# -------------------------------
years = [2025]
ytd_list = []

def rename_ytd_columns(df, prefix1 = "_p1", prefix2 = "_p2"):
    """
    Convert columns like:
        ytd_Aces_p1 → ytd_Aces
        ytd_Aces_p2 → ytd_Aces
    """
    new_cols = {}
    for col in df.columns:
        if col.endswith(prefix1) or col.startswith(prefix1):
            base = col.replace(prefix1, "")
            base = "winner_" + base
            new_cols[col] = base  # remove the _p1/_p2 suffix
        elif col.endswith(prefix2) or col.startswith(prefix2):
            base = col.replace(prefix2, "")
            base = "loser_" + base
            new_cols[col] = base  # remove the _p1/_p2 suffix
    return df.rename(columns=new_cols)

for year in years:
    ytd_df = process_year_data(year)
    if ytd_df is not None and len(ytd_df) > 0:

        # format names
        ytd_df["player1"] = ytd_df["player1"].astype(str).apply(format_player_name)
        ytd_df["player2"] = ytd_df["player2"].astype(str).apply(format_player_name)
        ytd_df["winner"]  = ytd_df["winner"].astype(str).apply(format_player_name)

        # rename columns cleanly
        ytd_df = rename_ytd_columns(ytd_df)

        ytd_list.append(ytd_df)

# Combine all years
YTD = pd.concat(ytd_list, ignore_index=True)

# Remove any columns that start with "match"
YTD = YTD[[col for col in YTD.columns if not col.startswith("match")]]

left_on  = ["Year", "Winner", "Loser", "Location"]
right_on = ["year", "player1", "player2", "location"]

rename_map = dict(zip(right_on, left_on))

YTD = YTD.rename(columns=rename_map)

print("Final YTD shape:", YTD.shape)
print("YTD example:\n", YTD.head())

# # -------------------------------
# # 2. Prepare Main DF
# # -------------------------------
# df = df.sort_values("Date").reset_index(drop=True)
# df["Year"] = '2025'

# # -------------------------------
# 3. Merge winner YTD stats
# player1 = winner, player2 = loser
# -------------------------------
# full_merged = pd.merge(
#     df,
#     YTD,
#     left_on=["Year", "Winner", "Loser", "Location"],
#     right_on=["year", "player1", "player2", "location"],
#     how="left"
# )

# # -------------------------------
# # 4. Drop join key duplicates
# # -------------------------------
# drop_cols = [col for col in full_merged.columns if col.endswith("_year") or col.endswith("_location")]
# df = full_merged.drop(columns=drop_cols, errors="ignore")

# print("Final merged df shape:", df.shape)



--- Processing Year 2025 ---
  Total for 2025: 3192 rows from 45 tournaments
Final YTD shape: (3192, 80)
YTD example:
                         round  duration                Winner seed1 scores1  \
0             Final - ESTADIO  01:37:55             Machac T.   (8)     7 6   
1        Semifinals - ESTADIO  02:03:53             Machac T.   (8)   6 1 6   
2        Semifinals - ESTADIO  02:15:45  Davidovich Fokina A.   NaN     7 7   
3  Quarterfinals - GRANDSTAND  01:38:03             Machac T.   (8)     6 7   
4     Quarterfinals - ESTADIO  02:10:51         Shapovalov D.   (9)   4 6 6   

                  Loser seed2  scores2                winner stats_url  ...  \
0  Davidovich Fokina A.   NaN    6 6 2             Machac T.       NaN  ...   
1          Nakashima B.   NaN    4 6 4             Machac T.       NaN  ...   
2         Shapovalov D.   (9)  6 3 6 1  Davidovich Fokina A.       NaN  ...   
3               Tien L.   (Q)      3 5             Machac T.       NaN  ...   
4         

In [3]:
new_df = pd.concat([df, YTD], ignore_index = True)
print(new_df.head(-5))

       ATP  Location              Tournament       Date  Series    Court  \
0      1.0  Brisbane  Brisbane International 2015-01-05  ATP250  Outdoor   
1      1.0  Brisbane  Brisbane International 2015-01-05  ATP250  Outdoor   
2      1.0  Brisbane  Brisbane International 2015-01-05  ATP250  Outdoor   
3      3.0      Doha  Qatar Exxon Mobil Open 2015-01-05  ATP250  Outdoor   
4      3.0      Doha  Qatar Exxon Mobil Open 2015-01-05  ATP250  Outdoor   
...    ...       ...                     ...        ...     ...      ...   
28046  NaN   Toronto                     NaN        NaT     NaN      NaN   
28047  NaN   Toronto                     NaN        NaT     NaN      NaN   
28048  NaN   Toronto                     NaN        NaT     NaN      NaN   
28049  NaN   Toronto                     NaN        NaT     NaN      NaN   
28050  NaN   Toronto                     NaN        NaT     NaN      NaN   

      Surface      Round  Best of         Winner  ...  \
0        Hard  1st Round      

In [4]:
sys.path.append('../src')

from elo_features import compute_elo_rankings

elo_df = compute_elo_rankings(new_df)
new_df = pd.merge(new_df, elo_df, left_index=True, right_index=True)

Elo rankings computing...
5000 matches computed...
10000 matches computed...
15000 matches computed...
20000 matches computed...
25000 matches computed...


In [5]:
from h2h import find_h2h

h2h_record, test_df = find_h2h(new_df)
